In [1]:
import torch
import torch.nn as nn

device ='cuda' if torch.cuda.is_available() else 'cpu'
print(device)
device = 'cpu'
block_size= 8
batch_size= 4
max_iters = 100000
learning_rate= 3e-4
eval_iters = 250
dropout = 0.2
n_embd =384
n_head =4
n_layer = 4 ## decoder block

cuda


In [2]:
with open('books.txt','r', encoding='utf-8') as f :
    text = f.read()
print(len(text))

728028


In [3]:
print(text[:500])

PRIDE.
                                  and
                               PREJUDICE

                                  by
                             Jane Austen,

                           with a Preface by
                           George Saintsbury
                                  and
                           Illustrations by
                             Hugh Thomson

                         [Illustration: 1894]

                       Ruskin       156. Charing
                      


In [4]:
chars= sorted(set(text))

In [5]:
print(chars)

['\n', ' ', '!', '&', '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '^', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '}', '·', 'à', 'â', 'é', 'ê', 'œ', '‘', '’', '“', '”']


In [6]:
vocab_size=len(chars)

In [7]:
string_to_int= { ch : i for i,ch in enumerate(chars)}
int_to_string= { i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

In [8]:
encoded_hello= encode('hello')
print(encoded_hello)
decoded_hello= decode(encoded_hello)
print(decoded_hello)

[60, 57, 64, 64, 67]
hello


In [9]:
data= torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([39, 40, 32, 27, 28,  9,  0,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1, 53, 66, 56,  0,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
         1,  1,  1,  1, 39, 40, 28, 33, 43, 27, 32, 26, 28,  0,  0,  1,  1,  1,
         1,  1,  1,  1,  1,  1,  1,  1,  1,  1])


In [10]:
n = int(0.8*len(data))
train_data= data[:n]
val_data = data[n:]

In [11]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size+1]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size , (batch_size,))
    #print(ix)
    x = torch.stack([data[i: i+ block_size] for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    x, y = x.to(device) , y.to(device)
    return x,y
x,y = get_batch('train')
print('input:')
print(x)

print('target:')
print(y)



# for t in range(block_size):
#     context = x[:t+1]
#     target = y[t]
#     print('when input is', context, 'target is', target)
    

input:
tensor([[ 1, 74, 57, 70, 77,  1, 64, 53],
        [ 1, 71, 67,  1, 71, 65, 53, 64],
        [77,  7, 90,  1, 55, 67, 66, 72],
        [70, 61, 67, 73, 71,  0, 55, 67]])
target:
tensor([[74, 57, 70, 77,  1, 64, 53, 72],
        [71, 67,  1, 71, 65, 53, 64, 64],
        [ 7, 90,  1, 55, 67, 66, 72, 61],
        [61, 67, 73, 71,  0, 55, 67, 66]])


In [12]:
randint= torch.randint(-100,100,(6,2))
randint

tensor([[ 75, -34],
        [-13, -59],
        [  7,  75],
        [ 64,   4],
        [ 20, -80],
        [-91, -91]])

In [13]:
tensor = torch.tensor([[0.1,1.3], [2,3]])
tensor

tensor([[0.1000, 1.3000],
        [2.0000, 3.0000]])

In [14]:
zeros= torch.zeros(2,3)
zeros

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [15]:
ones = torch.ones(4,3)
ones


tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])

In [16]:
input= torch.empty(2,3)
input

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 1.4013e-45, 0.0000e+00]])

In [17]:
arrange = torch.arange(8)
arrange

tensor([0, 1, 2, 3, 4, 5, 6, 7])

In [18]:
linspace = torch.linspace(3,10, steps=12)
linspace

tensor([ 3.0000,  3.6364,  4.2727,  4.9091,  5.5455,  6.1818,  6.8182,  7.4545,
         8.0909,  8.7273,  9.3636, 10.0000])

In [19]:
logspace = torch.logspace(start = -10, end= 5, steps = 5)
logspace

tensor([1.0000e-10, 5.6234e-07, 3.1623e-03, 1.7783e+01, 1.0000e+05])

In [20]:
eys= torch.eye(10)
eys

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

In [21]:
 a = torch.empty((3,3), dtype = torch.int64)
empty_like = torch.empty_like(a)
empty_like

tensor([[              0,               1,               1],
        [352951805673478,              80,             192],
        [106996294559256,             134,               0]])

In [22]:
probabilities = torch.tensor([0.1,0.7,0,2])
samples =torch.multinomial(probabilities, num_samples = 20 , replacement=True)
print(samples)

tensor([3, 3, 1, 3, 3, 3, 1, 0, 3, 1, 3, 3, 3, 3, 3, 3, 1, 3, 1, 1])


In [23]:
tensor =  torch.tensor([1,2,3,4])
out = torch.cat((tensor, torch.tensor([5])), dim = 0)
out

tensor([1, 2, 3, 4, 5])

In [24]:
out = torch.tril(torch.ones(5,5))
out

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

In [25]:
out = torch.triu(torch.ones(5,5))
out

tensor([[1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1.],
        [0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 1.]])

In [26]:
out = torch.zeros(5,5).masked_fill(torch.tril(torch.ones(5,5)) == 0, float('-inf'))# “scan through the msk matrix and when there is a cell with value 0, change the corresponding cell in source to inf”


out

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

In [27]:
torch.exp(out)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

In [28]:
input = torch.zeros(2,3,4)
out = input.transpose(0,2)
out.shape

torch.Size([4, 3, 2])

In [29]:
tensor1= torch.tensor([1 ,2,3])
tensor2= torch.tensor([4,5,6])
tensor3= torch.tensor([7,8,9])

stacked_tensor= torch.stack([tensor1,tensor2,tensor3])
stacked_tensor

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

# linear layer

In [30]:
import torch.nn as nn
sample = torch.tensor([10.,10.,10.])
linear = nn.Linear(3,3,bias = False)
print(linear(sample))

tensor([-2.1886,  2.5454,  0.6104], grad_fn=<SqueezeBackward4>)


# softmax

In [31]:
import torch.nn.functional as F

tensor4 = torch.tensor([1.0,2.4,4])

softmax_output = F.softmax(tensor4 , dim = 0)
print(softmax_output)

tensor([0.0398, 0.1613, 0.7989])


# Embedding

In [32]:
vocab_size = 80
embedding_dim= 6
embedding= nn.Embedding(vocab_size, embedding_dim)

input_indices = torch.LongTensor([1,5,4,2])
embedded_output = embedding(input_indices)

print(embedded_output.shape)
print(embedded_output)

torch.Size([4, 6])
tensor([[ 1.2517,  0.3778,  0.4964,  0.0242,  0.2571,  0.2460],
        [-1.4421,  0.3396, -0.2183,  0.7936,  0.1561, -1.3782],
        [ 0.3607,  1.0275,  0.5307, -0.9910,  0.4861, -0.8021],
        [ 1.4333, -0.5405, -0.9167,  1.2646,  1.6961,  0.1687]],
       grad_fn=<EmbeddingBackward0>)


In [33]:
a= torch.tensor([[3,3,2],[2,3,5],[3,4,2]])
b= torch.tensor([[3,5,2],[2,6,3],[1,1,1]])

print(a@b)
print(torch.matmul(a,b))

tensor([[17, 35, 17],
        [17, 33, 18],
        [19, 41, 20]])
tensor([[17, 35, 17],
        [17, 33, 18],
        [19, 41, 20]])


In [34]:
int_64 = torch.randint(2,(3,2)).float() # 0 to 1 range

float_32 = torch.rand(2,3)
print(int_64.dtype, float_32.dtype)

result = torch.matmul(int_64, float_32)
print(result)

torch.float32 torch.float32
tensor([[0.0000, 0.0000, 0.0000],
        [0.2287, 0.9039, 0.3218],
        [0.2008, 0.1530, 0.3041]])


# class

In [35]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

In [36]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [37]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [38]:
class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        y = self.sa(x)
        x = self.ln1(x + y)
        y = self.ffwd(x)
        x = self.ln2(x + y)
        return x

In [39]:
class LLM_model(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size , n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd,n_head = n_head) for _ in range(n_layer)])

        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    def forward(self,index,targets = None):
        B, T= index.shape
        #logits = self.token_embedding_table(index)
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(index) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets= targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    # def generate(self, index, max_new_tokens):
    #     for _ in range(max_new_tokens):
    #         logits, loss= self.forward(index)
    #         logits = logits[:,-1, :]
    #         probs= F.softmax(logits, dim= -1)
    #         index_next= torch.multinomial(probs,num_samples=1)
    #         index = torch.cat((index, index_next), dim=1)
    #     return index
    def generate(self, index, max_new_tokens):
            for _ in range(max_new_tokens):
                # Crop the sequence to the last 'block_size' tokens
                # This prevents T from ever exceeding your positional embedding limit
                index_cond = index[:, -block_size:]
                
                # Pass the cropped sequence to get the predictions
                logits, loss = self.forward(index_cond)
                
                # Focus only on the last time step
                logits = logits[:, -1, :]
                
                # Get probabilities
                probs = F.softmax(logits, dim=-1)
                
                # Sample the next token
                index_next = torch.multinomial(probs, num_samples=1)
                
                # Append the sampled index to the running sequence
                index = torch.cat((index, index_next), dim=1)
                
            return index
vocab_size = len(chars) # Reset vocab_size to the true number of unique characters (91)
model = LLM_model(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype= torch.long, device=device)#int 64 = long

generated_chars= decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

        


(bNd’:?}[·h]k?44f·4Op·hhi“xgmZXe53fIM9!pkBB“G8D^}_J“^y2TjU9YX^Lméê9àqEtx_sg3!_75]·;^l!^PzzG9g7_?_UéSh
Dj9g6l)D!w.CeœpàœcZUWJUx/mzf1V[;bf/r^6sbaœg2U
7:‘/;l}pf}7YS6D*L?GpulmsâGp·”[oj_·gTB?w-gbp&O;éCyK5SRNé;5T :uLt!Yé}V6’9à,f·TZ9t(bE *dsu_3“^Kp&87g“D.)9pPkzéKIosJuXI;(0·ê]64TO2‘wnuAY”é2·(T3-Bg,M?oC( wBo5km[x2{I&jw!t:bMêoyfLUFli“^·w!apâTYZ9FtRdSs;·:Dx4A. ”ZRjvzyeMvRb’o
2*4HZ_tfOqfdaqrr[^AfCYYAœ?wlfi(c”KqBH3tvBeœlpH8(VBsjvW!ow0oAi9g·‘yê7lq·4,6V7L‘xA9Y{_7FMm_”fl64OY·?wf-[bT,4c·sKsu)q_œ.OléEy3FtUD(aU&;T


# Class tutorial

In [40]:
import turtle


class polygon:
    def __init__(self, sides,name = "bugg"):
        self.sides = sides
        self.name = name
    def draw(self):
        for i in range(self.sides):
            turtle.forward(100)
            angle =((self.sides-2)*180)/self.sides
            angle = 180 -angle 
            
            turtle.right(angle)
        print(180-angle)
        #turtle.done()  

#square = polygon(4,"square")
#print(square.sides)
#print(square.name)
#polygon(5, "gambo").draw()

class Square(polygon):
    def __init__(self):
        super().__init__(4,name= "it's square")

    def draw(self):
        turtle.begin_fill()
        super().draw()
        turtle.end_fill()


#print(Square().draw())


In [41]:
import matplotlib.pyplot as plt

class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    def __add__(self, other):
        # I completed this method for you!
        # It adds the x's and y's together, then creates a new Point.
        x = self.x + other.x
        y = self.y + other.y
        return Point(x, y)
        
    def plot(self):
        plt.scatter(self.x, self.y)

a = Point(1,1)
b = Point(2,2)
c = a + b
print(c.x ,c.y)
# Optional: To actually see the new point 'c' on a graph, you can add:
# c.plot()
# plt.show()

3 3


In [42]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train','val']:
        losses = torch.zeros(eval_iters)
        for k in range (eval_iters):
            X, Y = get_batch(split)
            logits, loss= model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out
    
    

# Continue LLM _end of class tutorial

## optimizer and train part

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr= learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: { iter}, train loss {losses['train']:.4f}, val loss :{losses['val']:.4f}")
    xb,yb = get_batch('train')
    logits,loss = model.forward(xb,yb)
    optimizer.zero_grad(set_to_none= True)
    loss.backward()
    optimizer.step()
print(loss.item())


step: 0, train loss 4.5557, val loss :4.5585
step: 250, train loss 2.6017, val loss :2.5674
step: 500, train loss 2.4525, val loss :2.5039
step: 750, train loss 2.3678, val loss :2.3660
step: 1000, train loss 2.3107, val loss :2.3039
step: 1250, train loss 2.2435, val loss :2.2920
step: 1500, train loss 2.2794, val loss :2.2859
step: 1750, train loss 2.2298, val loss :2.2206
step: 2000, train loss 2.1813, val loss :2.2002
step: 2250, train loss 2.1608, val loss :2.1440
step: 2500, train loss 2.2017, val loss :2.1606
step: 2750, train loss 2.1090, val loss :2.1569
step: 3000, train loss 2.1042, val loss :2.1311
step: 3250, train loss 2.1253, val loss :2.1225
step: 3500, train loss 2.0961, val loss :2.1147
step: 3750, train loss 2.1151, val loss :2.0734
step: 4000, train loss 2.0976, val loss :2.0989
step: 4250, train loss 2.0556, val loss :2.0459
step: 4500, train loss 2.0919, val loss :2.1025
